# மாடல் கான்டெக்ஸ்ட் புரோட்டோகால் (MCP) ஆதரவுடன் Azure AI ஏஜென்ட்கள் - பைதான்

இந்த நோட்புக் பைதானில் மாடல் கான்டெக்ஸ்ட் புரோட்டோகால் (MCP) கருவிகளுடன் Azure AI ஏஜென்ட்களை எப்படி பயன்படுத்துவது என்பதை விளக்குகிறது. முக்கியமற்ற அங்கீகாரத்தைப் பயன்படுத்தி மேம்பட்ட திறன்களுக்காக வெளிப்புற MCP சர்வர்களை (Microsoft Learn போன்றவை) பயன்படுத்தக்கூடிய ஒரு புத்திசாலி ஏஜென்டை உருவாக்குவது எப்படி என்பதை இது காட்டுகிறது.


## தேவையான பைதான் தொகுப்புகளை நிறுவுதல்

முதலில், தேவையான பைதான் தொகுப்புகளை நிறுவ வேண்டும்:
- **azure-ai-projects**: முக்கிய Azure AI Projects SDK
- **azure-ai-agents**: ஏஜென்டுகளை உருவாக்க மற்றும் மேலாண்மை செய்ய Azure AI Agents SDK
- **azure-identity**: DefaultAzureCredential பயன்படுத்தி முக்கியமற்ற அங்கீகாரம் வழங்குகிறது
- **mcp**: பைதான் பயன்பாட்டிற்கான Model Context Protocol செயல்பாடு


## கீ இல்லா அங்கீகாரத்தின் நன்மைகள்

இந்த நோட்புக் **கீ இல்லா அங்கீகாரம்** பற்றிய விளக்கத்தை வழங்குகிறது, இது பல நன்மைகளை வழங்குகிறது:
- ✅ **API கீகளை நிர்வகிக்க தேவையில்லை** - Azure அடையாள அடிப்படையிலான அங்கீகாரத்தை பயன்படுத்துகிறது
- ✅ **மேம்பட்ட பாதுகாப்பு** - குறியீடு அல்லது கட்டமைப்பு கோப்புகளில் ரகசியங்கள் சேமிக்கப்படுவதில்லை
- ✅ **தானியங்கி சான்றிதழ் சுழற்சி** - Azure சான்றிதழ் வாழ்க்கைச்சுழற்சி மேலாண்மையை கையாளுகிறது
- ✅ **பங்கு அடிப்படையிலான அணுகல் கட்டுப்பாடு** - நுணுக்கமான அனுமதிகளுக்காக Azure RBAC ஐ பயன்படுத்துகிறது
- ✅ **பல சூழல்களுக்கு ஆதரவு** - மேம்பாடு மற்றும் உற்பத்தி ஆகியவற்றில் தானாகவே வேலை செய்கிறது

`DefaultAzureCredential` சிறந்த கிடைக்கக்கூடிய சான்றிதழ் மூலத்தை தானாகவே தேர்ந்தெடுக்கிறது:
1. **Managed Identity** (Azure இல் இயங்கும் போது)
2. **Azure CLI** சான்றிதழ்கள் (உள்ளூர் மேம்பாட்டின் போது)
3. **Visual Studio** சான்றிதழ்கள்
4. **சுற்றுப்புற மாறிகள்** (அமைக்கப்பட்டிருந்தால்)
5. **இணைய உலாவி மூலம் இடைமுக அங்கீகாரம்** (மாற்று வழியாக)


## கீ இல்லா அங்கீகார அமைப்பு

**கீ இல்லா அங்கீகாரத்திற்கான முன் தேவைகள்:**

### உள்ளூர் மேம்பாட்டிற்காக:
```bash
# Install Azure CLI and login
az login
# Verify your identity
az account show
```

### Azure சூழல்களுக்கு:
- உங்கள் Azure வளத்தில் **System-assigned Managed Identity**-ஐ இயக்கவும்
- மேலாண்மை அடிப்படையிலான சரியான **RBAC பங்குகளை** மேலாண்மை அடையாளத்திற்கு ஒதுக்கவும்:
  - Azure OpenAI அணுகலுக்காக `Cognitive Services OpenAI User`
  - Azure AI Projects அணுகலுக்காக `AI Developer`

### சூழல் மாறிகள் (விருப்பம்):
```python
# These are automatically detected by DefaultAzureCredential
# AZURE_CLIENT_ID=<your-client-id>
# AZURE_CLIENT_SECRET=<your-client-secret>
# AZURE_TENANT_ID=<your-tenant-id>
```

**API விசைகள் அல்லது இணைப்பு சரங்கள் தேவையில்லை!** 🔐


In [ ]:
! pip install azure-ai-projects -U
! pip install azure-ai-agents==1.1.0b4 -U
! pip install azure-identity -U
! pip3 install mcp==1.11.0 -U

## தேவையான நூலகங்களை இறக்குமதி செய்க

தேவையான Python மாட்யூல்களை இறக்குமதி செய்யவும்:
- **os, time**: சூழல் மாறிகள் மற்றும் தாமதங்களைச் செய்ய Python இன் நிலையான நூலகங்கள்
- **AIProjectClient**: Azure AI Projects க்கான முக்கிய கிளையன்ட்
- **DefaultAzureCredential**: Azure சேவைகளுக்கான விசையில்லா அங்கீகாரம்
- **MCP தொடர்பான வகைகள்**: MCP கருவிகளை உருவாக்க மற்றும் மேலாண்மை செய்யவும், அங்கீகாரங்களை கையாளவும்


In [ ]:
import os, time
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import McpTool, RequiredMcpToolCall, SubmitToolApprovalAction, ToolApproval


## MCP சர்வர் அமைப்புகளை உள்ளமைக்கவும்

சுற்றுச்சூழல் மாறிகள் மற்றும் பின்வாங்கல் இயல்புநிலை அமைப்புகளைப் பயன்படுத்தி MCP சர்வர் கட்டமைப்பை அமைக்கவும்:
- **MCP_SERVER_URL**: MCP சர்வரின் URL (இயல்புநிலை Microsoft Learn API)
- **MCP_SERVER_LABEL**: MCP சர்வரை அடையாளம் காண ஒரு லேபல் (இயல்புநிலை "mslearn")

இந்த அணுகுமுறை பல்வேறு சூழல்களில் நெகிழ்வான கட்டமைப்பை வழங்குகிறது.


In [ ]:
mcp_server_url = os.environ.get("MCP_SERVER_URL", "https://learn.microsoft.com/api/mcp")
mcp_server_label = os.environ.get("MCP_SERVER_LABEL", "mslearn")

## Azure AI Project கிளையண்டை உருவாக்குதல் (கீலற்ற அங்கீகாரம்)

Azure AI Project கிளையண்டை **கீலற்ற அங்கீகாரம்** மூலம் தொடங்குங்கள்:
- **endpoint**: Azure AI Foundry திட்டத்தின் endpoint URL
- **credential**: `DefaultAzureCredential()` ஐ பயன்படுத்தி பாதுகாப்பான, கீலற்ற அங்கீகாரம்
- **API கீக்கள் தேவையில்லை**: கிடைக்கக்கூடிய சிறந்த அங்கீகாரத்தை தானாகவே கண்டறிந்து பயன்படுத்துகிறது

**அங்கீகார செயல்முறை:**
1. நிர்வகிக்கப்பட்ட அடையாளத்தைச் (Managed Identity) தேடுகிறது (Azure சூழல்களில்)
2. Azure CLI அங்கீகாரத்தை (உள்ளூர் மேம்பாட்டிற்காக) பயன்படுத்துகிறது
3. தேவையான பிற அங்கீகார மூலங்களை பயன்படுத்துகிறது

இந்த அணுகுமுறை உங்கள் கோடில் API கீக்கள் அல்லது இணைப்பு strings ஐ நிர்வகிக்க தேவையற்றது.


In [ ]:
project_client = AIProjectClient(
    endpoint="Your Azure AI Foundry Endpoint",
    credential=DefaultAzureCredential(),
)

## MCP கருவி வரையறையை உருவாக்கவும்

Microsoft Learn MCP சேவையகத்துடன் இணைக்கும் MCP கருவியை உருவாக்கவும்:
- **server_label**: MCP சேவையகத்திற்கான அடையாளம்
- **server_url**: MCP சேவையகத்தின் URL முடுக்கம்
- **allowed_tools**: பயன்படுத்தக்கூடிய கருவிகளை கட்டுப்படுத்த விருப்பமான பட்டியல் (காலியான பட்டியல் அனைத்து கருவிகளையும் அனுமதிக்கும்)

இந்த கருவி முகவருக்கு Microsoft Learn ஆவணங்கள் மற்றும் வளங்களை அணுக உதவும்.


In [ ]:
mcp_tool = McpTool(
    server_label=mcp_server_label,
    server_url=mcp_server_url,
    allowed_tools=[],  # Optional: specify allowed tools
)


## ஏஜென்ட் உருவாக்கி உரையாடலை செயல்படுத்துதல் (கீலெஸ் வேலைப்பாடு)

இந்த விரிவான பகுதி முழுமையான **கீலெஸ் ஏஜென்ட் வேலைப்பாட்டை** விளக்குகிறது:

1. **AI ஏஜென்ட் உருவாக்குதல்**: GPT-4.1 நானோ மாடல் மற்றும் MCP கருவிகளைப் பயன்படுத்தி ஏஜென்டை அமைக்கவும்
2. **த்ரெட்பை உருவாக்குதல்**: தொடர்பு கொள்ள உரையாடல் த்ரெட்பை நிறுவவும்
3. **செய்தி அனுப்புதல்**: Azure OpenAI மற்றும் OpenAI இடையேயான வேறுபாடுகளை ஏஜென்டிடம் கேட்கவும்
4. **கருவி ஒப்புதல்களை கையாளுதல்**: தேவையான போது MCP கருவி அழைப்புகளை தானாகவே ஒப்புதல் அளிக்கவும்
5. **செயல்பாட்டை கண்காணித்தல்**: ஏஜென்டின் முன்னேற்றத்தை கண்காணித்து தேவையான நடவடிக்கைகளை மேற்கொள்ளவும்
6. **முடிவுகளை காட்டுதல்**: உரையாடல் மற்றும் கருவி பயன்பாட்டு விவரங்களை காட்டவும்

**கீலெஸ் அம்சங்கள்:**
- ✅ **கடினமாக குறியிடப்பட்ட ரகசியங்கள் இல்லை** - அனைத்து அங்கீகாரமும் Azure அடையாளத்தால் கையாளப்படுகிறது
- ✅ **இயல்பாக பாதுகாப்பானது** - பங்கு அடிப்படையிலான அணுகல் கட்டுப்பாட்டைப் பயன்படுத்துகிறது
- ✅ **எளிமையான பிரயோகம்** - அங்கீகார மேலாண்மை தேவையில்லை
- ✅ **தணிக்கைக்கு உகந்தது** - அனைத்து அணுகலும் Azure அடையாளத்தின் மூலம் கண்காணிக்கப்படுகிறது

ஏஜென்ட் MCP கருவிகளை Microsoft Learn வளங்களை அணுகுவதற்கு முழு பாதுகாப்புடன் மற்றும் API கீ மேலாண்மை இல்லாமல் பயன்படுத்தும்.


In [ ]:
with project_client:
    agents_client = project_client.agents

    # Create a new agent with keyless authentication
    # NOTE: To reuse existing agent, fetch it with get_agent(agent_id)
    agent = agents_client.create_agent(
        model="Your Azure OpenAI Model Deployment Name",
        name="my-mcp-agent",
        instructions="You are a helpful agent that can use MCP tools to assist users. Use the available MCP tools to answer questions and perform tasks.",
        tools=mcp_tool.definitions,
    )
    print(f"Created agent, ID: {agent.id}")
    print(f"MCP Server: {mcp_tool.server_label} at {mcp_tool.server_url}")

    # Create thread for communication
    thread = agents_client.threads.create()
    print(f"Created thread, ID: {thread.id}")

    # Create message to thread
    message = agents_client.messages.create(
        thread_id=thread.id,
        role="user",
        content="What's difference between Azure OpenAI and OpenAI?",
    )
    print(f"Created message, ID: {message.id}")

    # KEYLESS APPROACH: Handle tool approvals without hardcoded secrets
    
    # Option 1: Completely keyless (recommended for Azure identity-enabled MCP servers)
    # run = agents_client.runs.create(thread_id=thread.id, agent_id=agent.id, tool_resources=mcp_tool.resources)
    
    # Option 2: With minimal headers (if MCP server requires specific headers)
    # For demonstration purposes, using a placeholder header
    mcp_tool.update_headers("SuperSecret", "123456")  # Replace with actual auth if needed
    
    # Set approval mode - uncomment next line to disable approval requirement completely
    # mcp_tool.set_approval_mode("never")  # Fully automated, no approval needed
    
    run = agents_client.runs.create(thread_id=thread.id, agent_id=agent.id, tool_resources=mcp_tool.resources)
    print(f"Created run, ID: {run.id}")

    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)

        if run.status == "requires_action" and isinstance(run.required_action, SubmitToolApprovalAction):
            tool_calls = run.required_action.submit_tool_approval.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                agents_client.runs.cancel(thread_id=thread.id, run_id=run.id)
                break

            tool_approvals = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredMcpToolCall):
                    try:
                        print(f"Approving tool call: {tool_call}")
                        
                        # KEYLESS APPROVAL OPTIONS:
                        
                        # Option 1: No headers (fully keyless)
                        # tool_approvals.append(
                        #     ToolApproval(
                        #         tool_call_id=tool_call.id,
                        #         approve=True,
                        #         headers={}  # No headers needed for keyless
                        #     )
                        # )
                        
                        # Option 2: With headers (if MCP server requires them)
                        tool_approvals.append(
                            ToolApproval(
                                tool_call_id=tool_call.id,
                                approve=True,
                                headers=mcp_tool.headers,  # Uses configured headers if needed
                            )
                        )
                    except Exception as e:
                        print(f"Error approving tool_call {tool_call.id}: {e}")

            print(f"tool_approvals: {tool_approvals}")
            if tool_approvals:
                agents_client.runs.submit_tool_outputs(
                    thread_id=thread.id, run_id=run.id, tool_approvals=tool_approvals
                )

        print(f"Current run status: {run.status}")

    print(f"Run completed with status: {run.status}")
    if run.status == "failed":
        print(f"Run failed: {run.last_error}")

    # Display run steps and tool calls
    run_steps = agents_client.run_steps.list(thread_id=thread.id, run_id=run.id)

    # Loop through each step
    for step in run_steps:
        print(f"Step {step['id']} status: {step['status']}")

        # Check if there are tool calls in the step details
        step_details = step.get("step_details", {})
        tool_calls = step_details.get("tool_calls", [])

        if tool_calls:
            print("  MCP Tool calls:")
            for call in tool_calls:
                print(f"    Tool Call ID: {call.get('id')}")
                print(f"    Type: {call.get('type')}")

        print()  # add an extra newline between steps

    # Fetch and log all messages
    messages = agents_client.messages.list(thread_id=thread.id)
    print("\nConversation:")
    print("-" * 50)
    for msg in messages:
        if msg.text_messages:
            last_text = msg.text_messages[-1]
            print(f"{msg.role.upper()}: {last_text.text.value}")
            print("-" * 50)

    # Example of dynamic tool management (keyless)
    print(f"\nDemonstrating keyless dynamic tool management:")
    print(f"Current allowed tools: {mcp_tool.allowed_tools}")
    print("✅ All operations completed using keyless authentication!")


---

**குறிப்பு**:  
இந்த ஆவணம் [Co-op Translator](https://github.com/Azure/co-op-translator) என்ற AI மொழிபெயர்ப்பு சேவையைப் பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சிக்கின்றோம், ஆனால் தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளவும். அதன் தாய்மொழியில் உள்ள மூல ஆவணம் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்தவொரு தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பல்ல.
